In [ ]:
# Imports
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load features
Set `csv_path` to your features CSV (same as before).

In [ ]:
# Path to precomputed features (keep same preprocessing source)
import pandas as pd # Added import statement for pandas
csv_path = '/content/drive/MyDrive/resnet_features.csv'  # update if running locally
df = pd.read_csv(csv_path)
print('Shape:', df.shape)
df.head()

Shape: (3898, 2049)


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,label
0,0.039164,0.617642,0.107853,0.376942,0.089375,0.042309,0.135151,0.246696,0.143043,0.216616,...,0.265314,0.518185,0.623599,0.150417,0.017330,0.218101,0.447609,0.877618,0.249591,AD
1,0.253862,0.289451,0.183565,0.503103,0.000000,0.039847,0.640761,0.352711,0.710503,0.498195,...,0.012770,0.620834,0.281454,0.323947,0.003298,0.073039,0.590564,1.009496,1.523529,AD
2,0.167927,0.537961,0.123549,0.713740,0.080809,0.513323,0.790639,0.229424,0.334899,0.273876,...,0.010286,0.851536,0.294417,0.066262,0.002479,0.108432,0.983083,0.705193,1.205853,AD
3,0.155322,0.372970,0.098443,0.384186,0.000000,0.006767,0.399594,0.515298,0.849680,0.412460,...,0.022496,0.642848,0.213399,0.262902,0.013065,0.014067,0.698723,0.844151,1.481523,AD
4,0.120427,0.113370,0.089176,0.423628,0.000719,0.041530,0.666011,0.591804,0.625362,0.500816,...,0.018377,0.751583,0.114963,0.364352,0.025591,0.002929,0.731588,1.326745,1.401907,AD


In [ ]:
# Preprocess image features: ensure numeric and impute missing values (treat zeros as missing)
import numpy as np

# Identify feature columns (exclude 'label' if present)
if 'label' in df.columns:
    feature_cols = [c for c in df.columns if c != 'label']
else:
    feature_cols = df.columns.tolist()

# Coerce to numeric where possible (non-numeric become NaN)
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce')

# Identify numeric features after coercion
numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
non_numeric_dropped = sorted(set(feature_cols) - set(numeric_cols))

# Keep only numeric features + label (if present)
if 'label' in df.columns:
    df = pd.concat([df[numeric_cols], df[['label']]], axis=1)
else:
    df = df[numeric_cols]

# Replace inf/-inf with NaN in numeric columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Treat zeros as missing values in numeric features
zeros_before = int((df[numeric_cols] == 0).sum().sum())
df[numeric_cols] = df[numeric_cols].mask(df[numeric_cols] == 0, np.nan)

# Drop columns that are entirely NaN (after zero->NaN)
all_nan_cols = [c for c in numeric_cols if df[c].isna().all()]
if all_nan_cols:
    df.drop(columns=all_nan_cols, inplace=True)
    numeric_cols = [c for c in numeric_cols if c not in all_nan_cols]

# Count missing before fill
missing_before = int(df[numeric_cols].isna().sum().sum())

# Fill remaining NaN with column medians
medians = df[numeric_cols].median()
df[numeric_cols] = df[numeric_cols].fillna(medians)

missing_after = int(df[numeric_cols].isna().sum().sum())

print("Preprocessing complete.")
print(f"Numeric feature columns: {len(numeric_cols)}")
if non_numeric_dropped:
    print(f"Dropped non-numeric columns: {len(non_numeric_dropped)}")
if all_nan_cols:
    print(f"Dropped all-NaN columns: {len(all_nan_cols)}")
print(f"Zeros treated as missing: {zeros_before}")
print(f"Missing values: {missing_before} -> {missing_after}")

Preprocessing complete.
Numeric feature columns: 2048
Zeros treated as missing: 51906
Missing values: 51906 -> 0


In [ ]:
#df.head()
df.shape
preprocessed_csv_path = '/content/drive/MyDrive/resnet_features_preprocessed.csv'
try:
    df.to_csv(preprocessed_csv_path, index=False)
    print(f"Saved preprocessed CSV to: {preprocessed_csv_path}")
except Exception as e:
    fallback = 'resnet_features_preprocessed.csv'
    df.to_csv(fallback, index=False)
    print(f"Drive save failed ({e}); saved locally to: {fallback}")

Saved preprocessed CSV to: /content/drive/MyDrive/resnet_features_preprocessed.csv
